# Decision assessment (v3) — validation notebook

Everything behind the v3 report (`mra decide <engagement> --pdf …`) for one engagement, using the same `mra`
package. Change `ENGAGEMENT` to analyse your own; the default is the synthetic TechCo example
(`examples/build_techco.py`).

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from mra.assess import assess
from mra.explain import flip_sentence, outcome_label, signal_label
from mra.recommend import next_steps

pd.set_option("display.precision", 2, "display.width", 160, "display.max_colwidth", 90)
ENGAGEMENT = Path("examples/techco/engagement.yaml")
a = assess(ENGAGEMENT)
ins, ev = a.engagement.instrument, a.evidence
for p in a.problems:
    print("!", p)
for d, r in a.results.items():
    print(f"{d}: {outcome_label(ins, d, r.verdict)} — {r.status} (p = {r.probability:.2f})")

## 1. Respondents and current state

In [ ]:
print(ev.current_state)
a.engagement.respondents["role"].value_counts().rename("respondents").to_frame()

## 2. Evidence: sub-areas and axes

Point estimate and 95% bootstrap range. Sub-areas combine fact bands (weight 2) and survey perception (weight 1).

In [ ]:
rows = []
for name, v in ev.signals.items():
    if name.split(".")[0] in ("need", "capability", "fit", "requirement", "locality"):
        if not np.isnan(v[0]):
            rows.append({"signal": name, "label": signal_label(name, ins), "point": v[0],
                         "low": np.nanpercentile(v, 2.5), "high": np.nanpercentile(v, 97.5)})
pd.DataFrame(rows).set_index("signal").sort_index()

In [ ]:
pd.DataFrame(ev.divergences)   # perception vs measurement, > 1 point apart

## 3. Outcome distribution, explanations and preconditions

In [ ]:
for d, r in a.results.items():
    print(f"\n== {d}")
    print(pd.Series(r.distribution).rename("share of draws").to_string())
    for f in r.what_would_change:
        print("  ", flip_sentence(f, d, ins))
    for g, info in r.gates.get("required", {}).items():
        print(f"   gate {g}: {info['status']}  {info['checks']}  {'; '.join(info['corroboration'])}")
    if r.hard_stops or r.unmet_capabilities:
        print("   blocked by:", r.hard_stops, r.unmet_capabilities, "→ interim:", r.interim)

## 4. Hosting economics

In [ ]:
h = a.results.get("hosting")
if h and h.economics:
    display(pd.DataFrame({"ratio": h.economics["ratio"], "payback (from decision)": h.economics["payback_months"]}))
    print("horizons:", h.economics["horizons"])
    display(pd.DataFrame(h.economics["tornado"][1:]).assign(input=lambda d: d["input"].map(lambda f: signal_label("fact." + f, ins))))

## 5. Preferences, stakes and alignment

Group level only; groups below the minimum size are pooled or withheld.

In [ ]:
st = a.stakes
for d, p in st.preferences.items():
    print(f"\n{d}: preferred direction by group")
    display(pd.DataFrame(p["by_group"]).T.fillna(0))
pd.DataFrame(st.stake_alignment)

In [ ]:
for f in st.findings:
    print("-", f)

## 6. Next steps

In [ ]:
pd.DataFrame(next_steps(a))[["phase_label", "decision", "action", "owner", "reason"]]

## 7. Sensitivity to the rules

Re-run with a modified rule set and compare. The report always shows the verdict under the default rules
("shadow verdict") when an engagement adjusts them.

In [ ]:
import copy, yaml
from mra.decision import decide
rules = copy.deepcopy(ins.rules)
rules["evidence"]["confidence"] = 0.9
for d in a.results:
    r = decide(ev, rules, d)
    print(f"{d} at confidence 0.9: {r.verdict} (p = {r.probability:.2f})")